In [1]:
import numpy as np
import pandas as pd
import statsmodels.api as sm

Extraction des Outputs de 2022, dans un premiers temps on s'intéresse aux services d'Urgence, de MCO et de SSR. On pourra ajouter d'autres services par la suite (Psychiatrie par exemple)

In [19]:
FINESS = pd.read_excel("finess.xlsx")
FINESS = FINESS.drop(FINESS.columns[[1,2,4]],axis=1)
FINESS = FINESS.rename(columns={"FINESS":"FI","Statut Juridique":"Statut"})

SYGEN2022 = pd.read_csv("SAE/2022/SYGEN_2022r.csv", sep=";", encoding="latin-1")
Data = SYGEN2022[['FI','EFFSAL_TOT','EFFLIB_TOT','EFF_INFSANSSPE','EFF_INFAVECSPE','EFF_AID','EFF_DIR','EFF_DIRSOI','EFF_AUTADM','SEJHC_SSR','SEJHC_MCO','SEJHP_MCO','SEJ_HTP_TOT','VEN_HDJ_TOT','VEN_HDN_TOT']]
Data = Data.merge(FINESS, on="FI", how="left")

Urg2022 = pd.read_csv("SAE/2022/URGENCES2_2022r.csv", sep=";", encoding="latin-1")
Urg2022 = Urg2022[['FI','PASSU']]
Data = Data.merge(Urg2022, on="FI", how="left")

Total médecins (libéraux et saliariés confondus) : 

In [20]:
Data['MED'] = Data['EFFSAL_TOT'] + Data['EFFLIB_TOT']

Total infirmiers (spécialisés et non-spécialisés) : 

In [21]:
Data['IDE'] = Data['EFF_INFSANSSPE'] + Data['EFF_INFAVECSPE']

Total personnel administratif :

In [22]:
Data['ADMIN'] = Data['EFF_DIR'] + Data['EFF_DIRSOI'] + Data['EFF_AUTADM']

Total séjours MCO (hospitalisation partielle et complète confondu) :

In [23]:
Data['MCO'] = Data['SEJHC_MCO'] + Data['SEJHP_MCO']

Total séjours en psychiatrie (hospitalisations complètes, venues de jour et venues de nuit) :

In [24]:
Data['PSY'] = Data['SEJ_HTP_TOT'] + Data['VEN_HDJ_TOT'] + Data['VEN_HDN_TOT']

Passage des variables d'intérêt en log (convention log(0) = 0) :

In [29]:
Data["lURG"] = np.where(Data["PASSU"] > 0, np.log(Data["PASSU"]), 0)
Data["lMCO"] = np.where(Data["MCO"] > 0, np.log(Data["MCO"]), 0)
Data["lPSY"] = np.where(Data["PSY"] > 0, np.log(Data["PSY"]), 0)
Data["lSSR"] = np.where(Data["SEJHC_SSR"] > 0, np.log(Data["SEJHC_SSR"]), 0)

Data["lMED"] = np.where(Data["MED"] > 0, np.log(Data["MED"]), 0)
Data["lIDE"] = np.where(Data["IDE"] > 0, np.log(Data["IDE"]), 0)
Data["lAID"] = np.where(Data["EFF_AID"] > 0, np.log(Data["EFF_AID"]), 0)
Data["lADMIN"] = np.where(Data["ADMIN"] > 0, np.log(Data["ADMIN"]), 0)

On construit deux variables de contrôle : Statut_PNL (resp. Statut_PL) vaut 1 si l'établissement est privé non lucratif (resp. privé lucratif) 

In [30]:
dummies = pd.get_dummies(Data["Statut"], prefix="Statut")

Data["Statut_PNL"] = dummies["Statut_Privé non lucratif"]
Data["Statut_PL"]  = dummies["Statut_Privé lucratif"]

Data["Statut_PNL"] = Data["Statut_PNL"].astype(int)
Data["Statut_PL"]  = Data["Statut_PL"].astype(int)



Catégorisation des variables :

In [31]:
outputs = ["lURG", "lMCO", "lSSR", "lPSY"]
controls = ["Statut_PNL", "Statut_PL"]
inputs = ["lMED", "lIDE", "lAID", "lADMIN"]

Estimation des coefficients :

In [32]:
results = {}


for var in outputs:
    Data[f"{var}_PNL"] = Data[var] * Data["Statut_PNL"]
    Data[f"{var}_PL"]  = Data[var] * Data["Statut_PL"]


for inp in inputs:
    X = Data[outputs + controls + [f"{v}_PNL" for v in outputs] + [f"{v}_PL" for v in outputs]]
    X = sm.add_constant(X)
    y = Data[inp]
    
    model = sm.OLS(y, X).fit(cov_type="HC1")  # erreurs robustes
    results[inp] = model
    
    print("\n" + "="*60)
    print(f"Demande conditionnelle pour : {inp}")
    print(model.summary())
    
    # Interprétation en %
    for c in controls:
        coef = model.params[c]
        pct = (np.exp(coef) - 1) * 100
        print(f"{c} → différence d'utilisation : {pct:.2f}%")


Demande conditionnelle pour : lMED
                            OLS Regression Results                            
Dep. Variable:                   lMED   R-squared:                       0.445
Model:                            OLS   Adj. R-squared:                  0.443
Method:                 Least Squares   F-statistic:                     365.2
Date:                Fri, 30 Jan 2026   Prob (F-statistic):               0.00
Time:                        15:40:33   Log-Likelihood:                -6842.6
No. Observations:                4058   AIC:                         1.372e+04
Df Residuals:                    4043   BIC:                         1.381e+04
Df Model:                          14                                         
Covariance Type:                  HC1                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const          0

D'après les coefficients estimés, à production égale, un etablissement privé non lucratif emploie 3.35% de personnel en moins par rapport à un établissement public. Cela monte à 10.03% pour un établissement privé lucratif.